In [2]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [3]:

info = {
    'a':{
        're':0.12,
        'piotroski':8
    },
    'b':{
        're':0.18,
        'piotroski':4
    },
    'c':{
        're':0.09,
        'piotroski':9
    },
    'd':{
        're':0.15,
        'piotroski':6
    }
}


In [24]:
df = pd.DataFrame(info.values(),index=info.keys())
df

,re,piotroski
a,0.12,8
b,0.18,4
c,0.09,9
d,0.15,6


## 1 - GOAL PROGRAMMING

In [58]:
model = pyo.ConcreteModel()
 
model.ativos = pyo.Set(initialize= df.index)
model.re = pyo.Param(model.ativos, initialize=df[['re']])
model.piotroski = pyo.Param(model.ativos, initialize = df[['piotroski']])

#var
model.x = pyo.Var(model.ativos,bounds=(0,1) , domain=pyo.NonNegativeReals)

#quantos objetivos? 2 - retorno >= 14% , piotroski >=7  lembrar + o menos e - o mais
# entao sao dois "d"
model.obj1_menos = pyo.Var(domain=pyo.NonNegativeReals)
model.obj1_mais = pyo.Var(domain=pyo.NonNegativeReals)
model.obj2_menos = pyo.Var(domain=pyo.NonNegativeReals)
model.obj2_mais = pyo.Var(domain=pyo.NonNegativeReals)

# ojb ------------------------
def robj(model):
    return model.obj1_menos + model.obj2_menos
model.fobj = pyo.Objective(rule=robj, sense=pyo.minimize)

# restrições ------------------------
# soma 1
def soma1(model):
    return sum(model.x[a] for a in model.ativos) == 1
model.fsoma1 = pyo.Constraint(rule=soma1)

#rest objs 1 e 2
def r_re(model):
    return sum(model.x[a]*model.re[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais == 0.14
model.f_re = pyo.Constraint(rule=r_re)

def r_piotroski(model):
    return sum(model.x[a]*model.piotroski[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais == 7
model.fpiotroski = pyo.Constraint(rule=r_piotroski)

# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmp0lem1wdk.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmpv1r9kbto.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmpv1r9kbto.pyomo.lp
Objective sense      : Minimize
Variables            :       8  [Nneg: 4,  Box: 4]
Objective nonzeros   :       2
Linear constraints   :       3  [Equal: 3]
  Nonzeros           :      16
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min

In [60]:
model.display()

Model unknown

  Variables:
    x : Size=4, Index=ativos
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          a :     0 :  0.75 :     1 : False : False : NonNegativeReals
          b :     0 :  0.25 :     1 : False : False : NonNegativeReals
          c :     0 :   0.0 :     1 : False : False : NonNegativeReals
          d :     0 :   0.0 :     1 : False : False : NonNegativeReals
    obj1_menos : Size=1, Index=None
        Key  : Lower : Value                : Upper : Fixed : Stale : Domain
        None :     0 : 0.005000000000000018 :  None : False : False : NonNegativeReals
    obj1_mais : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   0.0 :  None : False : False : NonNegativeReals
    obj2_menos : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   0.0 :  None : False : False : NonNegativeReals
    obj2_mais : Size=1, Index=None
        Key  : Lower : V

## 2 - NSGA II

In [119]:
import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.core.repair import Repair


class Normaliza(Repair):
    def _do(self, problem, X, **kwargs):
        return X / X.sum(axis=1, keepdims=True)   # cada linha (individuo) dividida pela sua propria soma

class MyProblem(ElementwiseProblem):

    def __init__(self):
        super().__init__(n_var=4,
                         n_obj=2,
                        #  n_ieq_constr=0,
                        #  n_eq_constr=1,
                         xl=0,
                         xu=1)

    # COmo estou usando o REPAIR, nao precisa declarar a variavel que soma 1 pois     
    # def __init__(self):
    #     super().__init__(n_var=4,
    #                      n_obj=2,
    #                      n_ieq_constr=0,
    #                      n_eq_constr=1,
    #                      xl=0,
    #                      xu=1)

    def _evaluate(self, x, out, *args, **kwargs):

        f1 = sum(x[i]*df['re'].values[i] for i in range(len(info.keys())))
        f2 = sum(x[i]*df['piotroski'].values[i] for i in range(len(info.keys())))

        # As restrições precisam ser <= 0 
        # g1 = sum(x) - 1
        

        out["F"] = [-f1,-f2]
        # out["H"] = [g1]


problem = MyProblem()

algorithm = NSGA2(
    pop_size=80,
    repair=Normaliza()
    # n_offsprings=10,
    # sampling=FloatRandomSampling(),
    # crossover=SBX(prob=0.9, eta=15),
    # mutation=PM(eta=20),
    # eliminate_duplicates=True
)

termination = get_termination("n_gen", 80)

res = minimize(problem,
               algorithm,
               termination,
               seed=1,
               save_history=True,
               verbose=True)

X = res.X
F = res.F

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       80 |     28 |             - |             -
     2 |      160 |     48 |  0.0467171871 |         ideal
     3 |      240 |     69 |  0.0248205807 |         ideal
     4 |      320 |     78 |  0.0059097269 |             f
     5 |      400 |     80 |  0.0052793939 |         ideal
     6 |      480 |     80 |  0.0048877688 |             f
     7 |      560 |     80 |  0.0937669740 |         ideal
     8 |      640 |     80 |  0.0040001249 |         ideal
     9 |      720 |     80 |  0.0140153671 |         ideal
    10 |      800 |     80 |  0.0521039325 |         ideal
    11 |      880 |     80 |  0.0156558417 |         ideal
    12 |      960 |     80 |  0.0472441874 |         ideal
    13 |     1040 |     80 |  0.0018735312 |             f
    14 |     1120 |     80 |  0.0118911887 |         ideal
    15 |     1200 |     80 |  0.0197746436 |         ideal
    16 |     1280 |     80 |  0.0264602504 |         ide

In [120]:
print(X)
print('-------')
print(F)

[[2.18456977e-04 9.96133964e-01 2.41560755e-04 3.40601859e-03]
 [1.44933873e-04 3.45571111e-06 9.99293842e-01 5.57768599e-04]
 [1.57941459e-02 1.18905406e-02 9.63848257e-01 8.46705663e-03]
 [6.73882018e-01 8.91726439e-04 2.47522141e-02 3.00474042e-01]
 [1.55407012e-01 5.75156541e-04 8.43540681e-01 4.77150326e-04]
 [1.81080187e-01 6.64993727e-01 4.46647599e-02 1.09261326e-01]
 [1.09853622e-03 9.72391093e-01 9.38281130e-03 1.71275590e-02]
 [6.61521882e-01 6.82439514e-04 3.37034415e-01 7.61263128e-04]
 [3.95833224e-01 5.91687044e-01 1.06885507e-02 1.79118167e-03]
 [5.47701781e-01 3.92400769e-01 1.47894060e-02 4.51080437e-02]
 [1.77266800e-02 9.41503654e-01 8.65897146e-03 3.21106945e-02]
 [4.23227222e-01 5.59004272e-01 8.34825678e-05 1.76850233e-02]
 [4.09963790e-01 1.95535182e-01 1.27515518e-02 3.81749476e-01]
 [2.84436766e-01 2.26661450e-04 7.15313093e-01 2.34802402e-05]
 [1.29472414e-01 6.73839294e-01 9.74123316e-04 1.95714168e-01]
 [8.56298465e-01 1.42986785e-04 1.42788064e-01 7.704851

In [121]:
len(X)

80

In [129]:
print(X[3].round(4))
print(F[3])

[0.6739 0.0009 0.0248 0.3005]
[-0.12832516 -7.42023722]
